# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

In [ ]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [ ]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [ ]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [9]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def get_cat_fact():
  """Funkcja pobierająca losowy fakt o kotach z API."""
  return requests.get(CAT_API_URL, timeout=10).json().get('fact')

start = time.time()
facts_amount = 20
facts_sequential = []

for _ in range(facts_amount):
    fact = get_cat_fact()
    facts_sequential.append(fact)

sequential_time = time.time() - start

for i, fact in enumerate(facts_sequential, 1):
    print(f"{i}. {fact}")

print(f"\nCzas sekwencyjny: {sequential_time:.2f}s")

start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    facts_threaded = list(executor.map(lambda _: get_cat_fact(), range(facts_amount)))

threaded_time = time.time() - start
for i, fact in enumerate(facts_threaded, 1):
    print(f"{i}. {fact}")

print(f"\nCzas wielowątkowy: {threaded_time:.2f}s")

print(f"\nPorównanie: {sequential_time:.2f}s (sekw.) vs {threaded_time:.2f}s (wątki)")
print(f"Przyspieszenie: {sequential_time / threaded_time:.1f}x")

1. Edward Lear, author of \The Owl and the Pussycat\"", is said to have had his new house in San Remo built to exactly the same specification as his previous residence, so that his much-loved tabby, Foss, would immediately feel at home."""
2. In one stride, a cheetah can cover 23 to 26 feet (7 to 8 meters).
3. A cat has approximately 60 to 80 million olfactory cells (a human has between 5 and 20 million).
4. It may take as long as 2 weeks for a kitten to be able to hear well.  Their eyes usually open between 7 and 10 days, but sometimes it happens in as little as 2 days.
5. Unlike humans, cats are usually lefties. Studies indicate that their left paw is typically their dominant paw.
6. Retractable claws are a physical phenomenon that sets cats apart from the rest of the animal kingdom. I n the cat family, only cheetahs cannot retract their claws.
7. Purring does not always indicate that a cat is happy. Cats will also purr loudly when they are distressed or in pain.
8. A cat's whiskers 

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [10]:
import queue
import threading
import time

NUMBERS_COUNT = 20
SENTINEL = None

shared_queue = queue.Queue()
even_numbers = []
odd_numbers = []

def producer():
    for n in range(1, NUMBERS_COUNT + 1):
        shared_queue.put(n)
        print(f"[Producent] Dodano: {n}")
        time.sleep(0.05)
    shared_queue.put(SENTINEL)
    shared_queue.put(SENTINEL)

def consumer_even():
    while True:
        item = shared_queue.get()
        if item is SENTINEL:
            break
        if item % 2 == 0:
            even_numbers.append(item)
            print(f"  [Konsument PARZYSTY] Pobrał: {item}")
        else:
            shared_queue.put(item)
        shared_queue.task_done()

def consumer_odd():
    while True:
        item = shared_queue.get()
        if item is SENTINEL:
            break
        if item % 2 != 0:
            odd_numbers.append(item)
            print(f"  [Konsument NIEPARZYSTY] Pobrał: {item}")
        else:
            shared_queue.put(item)
        shared_queue.task_done()

t_producer = threading.Thread(target=producer)
t_even = threading.Thread(target=consumer_even)
t_odd = threading.Thread(target=consumer_odd)

t_producer.start()
t_even.start()
t_odd.start()

t_producer.join()
t_even.join()
t_odd.join()

print(f"\nParzyste ({len(even_numbers)}): {sorted(even_numbers)}")
print(f"Nieparzyste ({len(odd_numbers)}): {sorted(odd_numbers)}")

[Producent] Dodano: 1
  [Konsument NIEPARZYSTY] Pobrał: 1
[Producent] Dodano: 2
  [Konsument PARZYSTY] Pobrał: 2
[Producent] Dodano: 3
  [Konsument NIEPARZYSTY] Pobrał: 3
[Producent] Dodano: 4
  [Konsument PARZYSTY] Pobrał: 4
[Producent] Dodano: 5
  [Konsument NIEPARZYSTY] Pobrał: 5
[Producent] Dodano: 6
  [Konsument PARZYSTY] Pobrał: 6
[Producent] Dodano: 7
  [Konsument NIEPARZYSTY] Pobrał: 7
[Producent] Dodano: 8
  [Konsument PARZYSTY] Pobrał: 8
[Producent] Dodano: 9
  [Konsument NIEPARZYSTY] Pobrał: 9
[Producent] Dodano: 10
  [Konsument PARZYSTY] Pobrał: 10
[Producent] Dodano: 11
  [Konsument NIEPARZYSTY] Pobrał: 11
[Producent] Dodano: 12
  [Konsument PARZYSTY] Pobrał: 12
[Producent] Dodano: 13
  [Konsument NIEPARZYSTY] Pobrał: 13
[Producent] Dodano: 14
  [Konsument PARZYSTY] Pobrał: 14
[Producent] Dodano: 15
  [Konsument NIEPARZYSTY] Pobrał: 15
[Producent] Dodano: 16
  [Konsument PARZYSTY] Pobrał: 16
[Producent] Dodano: 17
  [Konsument NIEPARZYSTY] Pobrał: 17
[Producent] Dodano: 18

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [13]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    LIMIT = 10_000
    numbers = range(1, LIMIT + 1)
    cores = multiprocessing.cpu_count()

    # --- Sekwencyjnie ---
    start = time.time()
    results_seq = [calculate_power_sum(n) for n in numbers]
    seq_time = time.time() - start
    print(f"Sekwencyjnie: {seq_time:.2f}s")

    # --- Wieloprocesowo ---
    start = time.time()
    with multiprocessing.Pool(processes=cores) as pool:
        results_mp = pool.map(calculate_power_sum, numbers)
    mp_time = time.time() - start
    print(f"Wieloprocesowo ({cores} rdzeni): {mp_time:.2f}s")

    # --- Porównanie ---
    print(f"\nPrzyspieszenie: {seq_time / mp_time:.1f}x")
    print(f"Przykład: calculate_power_sum(5) = {results_mp[4]}")


Sekwencyjnie: 0.25s
Wieloprocesowo (14 rdzeni): 0.14s

Przyspieszenie: 1.8x
Przykład: calculate_power_sum(5) = 9860761315262647567646607066034827870915080438862787559628486633300780
